# Privacy-Preserving Federated Learning — Walkthrough

This notebook demonstrates the core components of our federated learning framework:
1. Data partitioning (IID vs non-IID)
2. FedAvg training across simulated clients
3. Differential privacy with DP-SGD
4. Secure aggregation simulation
5. Privacy budget tracking

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from torch.utils.data import Dataset, Subset

# Create a synthetic dataset for demonstration
class SyntheticMNIST(Dataset):
    """Synthetic dataset mimicking MNIST structure."""
    def __init__(self, n=2000, num_classes=10):
        self.data = torch.randn(n, 1, 28, 28)
        self.targets = torch.randint(0, num_classes, (n,)).tolist()
    
    def __len__(self):
        return len(self.targets)
    
    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx]

dataset = SyntheticMNIST(n=2000)
print(f'Dataset: {len(dataset)} samples, shape={dataset.data.shape}')

## 1. Data Partitioning: IID vs Non-IID

In federated learning, data is distributed across clients. The distribution can be:
- **IID**: Each client gets a random uniform sample (ideal case)
- **Non-IID**: Clients have skewed label distributions (realistic case)

We use Dirichlet allocation for non-IID partitioning. Lower alpha = more heterogeneous.

In [ ]:
from src.data.partitioner import partition_iid, partition_dirichlet, compute_heterogeneity

num_clients = 5

# IID partition
iid_part = partition_iid(dataset, num_clients)
print(f'IID heterogeneity: {compute_heterogeneity(iid_part):.4f}')
for s in iid_part.stats:
    print(f'  Client {s.client_id}: {s.num_samples} samples, {len(s.label_counts)} classes')

print()

# Non-IID partition (Dirichlet alpha=0.1)
noniid_part = partition_dirichlet(dataset, num_clients, alpha=0.1)
print(f'Non-IID (alpha=0.1) heterogeneity: {compute_heterogeneity(noniid_part):.4f}')
for s in noniid_part.stats:
    top = max(s.label_distribution, key=s.label_distribution.get)
    print(f'  Client {s.client_id}: {s.num_samples} samples, dominant class={top} ({s.label_distribution[top]*100:.0f}%)')

## 2. FedAvg Training

The server orchestrates training: select clients → broadcast model → local training → aggregate updates.

In [ ]:
from src.server import FederatedServer, ServerConfig
from src.models.simple_models import ModelConfig

config = ServerConfig(
    num_rounds=10,
    num_clients=5,
    clients_per_round=3,
    model_config=ModelConfig(model_type='mlp', input_size=28),
    partition_strategy='iid',
)

server = FederatedServer(config)
partition = server.setup_clients(dataset)
results = server.train(verbose=True)

# Plot convergence
losses = [r.avg_loss for r in results]
print(f'\nFinal loss: {losses[-1]:.4f}')

## 3. Non-IID Impact on Convergence

Non-IID data makes federated training harder. Let's compare IID vs non-IID convergence.

In [ ]:
# IID training
iid_config = ServerConfig(
    num_rounds=10, num_clients=5, clients_per_round=5,
    model_config=ModelConfig(model_type='mlp', input_size=28),
    partition_strategy='iid',
)
iid_server = FederatedServer(iid_config)
iid_server.setup_clients(dataset)
iid_results = iid_server.train(verbose=False)

# Non-IID training
noniid_config = ServerConfig(
    num_rounds=10, num_clients=5, clients_per_round=5,
    model_config=ModelConfig(model_type='mlp', input_size=28),
    partition_strategy='dirichlet', dirichlet_alpha=0.1,
)
noniid_server = FederatedServer(noniid_config)
noniid_server.setup_clients(dataset)
noniid_results = noniid_server.train(verbose=False)

print('Round | IID Loss | Non-IID Loss')
print('-' * 35)
for i in range(len(iid_results)):
    print(f'  {i:3d}  | {iid_results[i].avg_loss:.4f}   | {noniid_results[i].avg_loss:.4f}')

## 4. Privacy Budget Tracking

The privacy accountant tracks cumulative epsilon using RDP composition.

In [ ]:
from src.privacy.accountant import PrivacyAccountant, AccountantConfig

accountant = PrivacyAccountant(AccountantConfig(target_delta=1e-5))

noise_mult = 1.0
sample_rate = 0.01

print('Steps | Epsilon')
print('-' * 20)
for steps in [10, 50, 100, 500, 1000, 5000]:
    eps = accountant.get_epsilon_for_steps(noise_mult, sample_rate, steps)
    print(f'{steps:5d} | {eps:.4f}')

max_steps = accountant.max_steps_for_budget(noise_mult, sample_rate, target_epsilon=10.0)
print(f'\nMax steps for epsilon=10.0: {max_steps}')

## 5. Secure Aggregation

Clients mask their updates with pairwise random masks that cancel during summation. The server never sees individual updates.

In [ ]:
from src.aggregation.secure_agg import (
    SecureAggConfig, MaskedUpdate,
    mask_client_update, secure_aggregate, verify_mask_cancellation
)

n_clients = 4
sa_config = SecureAggConfig(num_clients=n_clients, threshold=3)
client_ids = list(range(n_clients))

# Simulate client weights
original = [{'w': torch.randn(10, 5)} for _ in range(n_clients)]
samples = [100, 200, 150, 250]

# Mask and aggregate
masked = [
    MaskedUpdate(cid, mask_client_update(cid, w, client_ids, sa_config), n)
    for cid, w, n in zip(client_ids, original, samples)
]

result = secure_aggregate(masked, sa_config)
correct = verify_mask_cancellation(original, samples, result)
print(f'Secure aggregation correct: {correct}')
print(f'Aggregated weight shape: {result["w"].shape}')

## 6. Noise Mechanisms

Differential privacy relies on calibrated noise. Let's visualize the noise-privacy tradeoff.

In [ ]:
from src.privacy.noise import (
    calibrate_gaussian_sigma, calibrate_laplace_scale,
    NoiseConfig, NoiseType, add_noise, clip_tensor
)

# Calibration across epsilon values
print('Epsilon | Gaussian sigma | Laplace scale')
print('-' * 45)
for eps in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    sigma = calibrate_gaussian_sigma(eps, 1e-5, sensitivity=1.0)
    scale = calibrate_laplace_scale(eps, sensitivity=1.0)
    print(f'  {eps:5.1f} | {sigma:14.4f} | {scale:13.4f}')

# Gradient clipping demo
grad = torch.randn(50) * 5  # Large gradient
clipped = clip_tensor(grad, max_norm=1.0)
print(f'\nOriginal grad norm: {torch.norm(grad):.4f}')
print(f'Clipped grad norm:  {torch.norm(clipped):.4f}')

## Summary

This framework demonstrates the key building blocks of privacy-preserving federated learning:

| Component | Purpose |
|-----------|--------|
| FedAvg | Weighted averaging of client model updates |
| FedProx | Proximal regularization for non-IID stability |
| DP-SGD | Per-sample gradient clipping + noise for privacy |
| Secure Aggregation | Pairwise masking so server can't see individual updates |
| Privacy Accountant | RDP composition to track cumulative privacy loss |
| Dirichlet Partitioning | Realistic non-IID data distribution simulation |